In [10]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================
# 1. Generate dataset
# =====================================================

dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):

        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

In [11]:
# =====================================================
# 2. Encode data
# =====================================================

def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x


inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


In [12]:
# =====================================================
# 3. Vanilla RNN model
# =====================================================

class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bias=False,
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [13]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

In [14]:
# =====================================================
# 4. Training setup
# =====================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [6]:
# =====================================================
# 5. Training loop
# =====================================================

epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.865568
Epoch [100/300] Loss = 0.652587
Epoch [150/300] Loss = 0.503802
Epoch [200/300] Loss = 0.424295
Epoch [250/300] Loss = 0.374586
Epoch [300/300] Loss = 0.334033


In [7]:
# =====================================================
# 6. Evaluation
# =====================================================

with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 0.9129999876022339


In [8]:
# =====================================================
# 7. Test on all possible transitions
# =====================================================

print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=A
Current Dish=B, Weather=Rainy --> Predicted Next Dish=B
Current Dish=C, Weather=Sunny --> Predicted Next Dish=C
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


Let's check the weights:

In [9]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[-0.2008, -1.9146,  1.7634,  2.8409, -0.3449],
        [-1.0696, -0.4272,  2.1457, -2.2777,  1.2596],
        [ 0.8022,  1.9007, -1.2879, -1.2476,  3.0857]])

rnn.weight_hh_l0
tensor([[ 0.1236,  1.0501, -1.4131],
        [-1.6269, -0.3027,  0.4574],
        [ 0.1520, -0.3078,  0.6381]])

fc.weight
tensor([[ 1.3966,  0.1675,  1.3419],
        [-2.1367, -2.0800,  1.0327],
        [-1.4313,  2.4620, -2.2554]])



## RNN Equations

![RNN](https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-unrolled.png)

Let

- $x_t$ = input at time step $t$
- $h_t$ = hidden state at time step $t$
- $y_t$ = output logits at time step $t$

Weights:

- $W_{ih}$ = input-to-hidden matrix
- $W_{hh}$ = hidden-to-hidden matrix
- $W_{fc}$ = hidden-to-output matrix

## Hidden State Update

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

But, with `bias=False`,
$$
\boxed{h_t = \tanh\left(W_{ih}x_t + W_{hh}h_{t-1}\right)}
$$

where,

$$
W_{ih} \in \mathbb{R}^{H \times D}
$$

$$
W_{hh} \in \mathbb{R}^{H \times H}
$$

$$
x_t \in \mathbb{R}^{D}
$$

$$
h_t \in \mathbb{R}^{H}
$$

For your model:

- $D = 5$ (A, B, C, Sunny, Rainy)
- $H = 3$

Thus

$$
W_{ih} \in \mathbb{R}^{3 \times 5}
$$

$$
W_{hh} \in \mathbb{R}^{3 \times 3}
$$

### Output Layer

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

But, for `bias=False`,

$$
\boxed{
y_t = W_{fc}h_t}
$$

where

$$
W_{fc} \in \mathbb{R}^{3 \times 3}
$$

and

$$
y_t \in \mathbb{R}^{3}
$$

The three output values are logits:

$$
y_t =
\begin{bmatrix}
s_A \\
s_B \\
s_C
\end{bmatrix}
$$

### Softmax

Convert logits into probabilities:

$$
p_i =
\frac{e^{s_i}}
{\sum_j e^{s_j}}
$$

or

$$
p(y_t = k)
=
\frac{e^{s_k}}
     {e^{s_A}+e^{s_B}+e^{s_C}}
$$

### Prediction

The predicted dish is

$$
\hat y_t
=
\arg\max_k p(y_t=k)
$$

which is equivalent to

$$
\hat y_t
=
\arg\max_k s_k
$$

because softmax preserves ordering.

### Explicit Form For Your Network

Input vector:

$$
x_t =
\begin{bmatrix}
A\\
B\\
C\\
Sunny\\
Rainy
\end{bmatrix}
$$

For example:

Dish = A, Weather = Rainy

$$
x_t =
\begin{bmatrix}
1\\
0\\
0\\
0\\
1
\end{bmatrix}
$$

### Hidden Unit Equations

Let

$$
h_t =
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

Then

$$
h_{1,t}
=
\tanh
\left(
w^{(1)}_{ih}x_t
+
w^{(1)}_{hh}h_{t-1}
\right)
$$

$$
h_{2,t}
=
\tanh
\left(
w^{(2)}_{ih}x_t
+
w^{(2)}_{hh}h_{t-1}
\right)
$$

$$
h_{3,t}
=
\tanh
\left(
w^{(3)}_{ih}x_t
+
w^{(3)}_{hh}h_{t-1}
\right)
$$

where each row of $W_{ih}$ and $W_{hh}$ corresponds to one hidden neuron.


### Output Logits

$$
s_A
=
w_A^\top h_t
$$

$$
s_B
=
w_B^\top h_t
$$

$$
s_C
=
w_C^\top h_t
$$

or

$$
\begin{bmatrix}
s_A\\
s_B\\
s_C
\end{bmatrix}
=
W_{fc}
\begin{bmatrix}
h_{1,t}\\
h_{2,t}\\
h_{3,t}
\end{bmatrix}
$$

### If Biases Were Enabled

PyTorch would compute

$$
h_t
=
\tanh
\left(
W_{ih}x_t
+
W_{hh}h_{t-1}
+
b_{ih}
+
b_{hh}
\right)
$$

and

$$
y_t
=
W_{fc}h_t
+
b_{fc}
$$

where

$$
b_{ih} \in \mathbb{R}^{H}
$$

$$
b_{hh} \in \mathbb{R}^{H}
$$

$$
b_{fc} \in \mathbb{R}^{3}
$$

For your current `bias=False` setup, all bias terms disappear.

___

References:

- https://docs.pytorch.org/docs/2.12/generated/torch.nn.RNN.html#torch.nn.RNN
- https://jaketae.github.io/study/pytorch-rnn/
- https://github.com/udacity/deep-learning-v2-pytorch/blob/master/recurrent-neural-networks/time-series/Simple_RNN.ipynb

# find out what the `hidden_size` means for an RNN Model.

It specifies the number of neurons (hidden units) in the RNN's hidden layer, which determines the size of the hidden state passed from one time step to the next.